# API DESIGN

# RestAPI

# we haev restful API standard

# RESTful API Standard

You're right - let me explain the RESTful API standard.

---

## What is REST?

**REST = Representational State Transfer**

**Definition:** An architectural style for designing networked applications, specifically APIs.

**Created by:** Roy Fielding in 2000 (in his PhD dissertation)

**Purpose:** A standardized way to build web APIs so that:
- Different systems can communicate easily
- APIs are predictable and consistent
- Developers understand APIs without documentation

---

## Core Principle: Resources

**Everything in REST is a "resource"**

**Resource:** Any piece of data or entity in your system

**Examples:**
- User
- Post
- Comment
- Product
- Order
- Book

**Each resource has:**
1. A unique identifier (ID)
2. A URL (endpoint)
3. Representations (usually JSON)

---

## REST Constraints (6 Rules)

### 1. Client-Server Architecture

**Separation of concerns:**
- Client handles user interface
- Server handles data storage and business logic
- They communicate over HTTP

**Why:** They can evolve independently

---

### 2. Stateless

**Each request contains everything needed:**
- Server doesn't remember previous requests
- No session state on server (or uses tokens)
- Every request is independent

**Example:**
```
Request 1: GET /users (includes auth token)
Request 2: GET /posts (includes auth token again)

Server doesn't remember Request 1 when processing Request 2
```

---

### 3. Cacheable

**Responses should indicate if they can be cached:**
- Improves performance
- Reduces server load

**Headers:**
```
Cache-Control: max-age=3600
Cache-Control: no-cache
ETag: "33a64df551425fcc55e4d42a148795d9"
```

---

### 4. Uniform Interface

**Standardized way to interact with resources:**
- Use HTTP methods correctly
- Use standard status codes
- Use consistent URL patterns
- Use standard data formats (JSON)

**This is the most important constraint for API design.**

---

### 5. Layered System

**Client doesn't know if connected directly to server or through intermediaries:**
- Load balancers
- Proxies
- Caches
- Gateways

**Example:**
```
Client → Load Balancer → API Server
Client → CDN → API Server
Client → API Gateway → Microservice

Client doesn't know the architecture
```

---

### 6. Code on Demand (Optional)

**Server can send executable code:**
- JavaScript
- Applets

**Rarely used in modern APIs.**

---

## RESTful API Design: The Standard

### Resource-Based URLs

**Structure: Collection → Resource**

```
/users              ← Collection of users
/users/123          ← Specific user (ID: 123)

/posts              ← Collection of posts
/posts/456          ← Specific post (ID: 456)

/users/123/posts    ← Posts by user 123
/posts/456/comments ← Comments on post 456
```

**Rules:**
- Use **nouns**, not verbs
- Use **plural** for collections
- Use **IDs** to identify specific resources

---

**❌ Bad (not RESTful):**
```
/getUsers           ← Verb in URL
/user               ← Singular
/users/delete/123   ← Action in URL
/createPost         ← Verb in URL
```

**✅ Good (RESTful):**
```
/users              ← Noun, plural
/users/123          ← Resource with ID
/posts              ← Noun, plural
/posts/456          ← Resource with ID
```

---

### HTTP Methods (Verbs)

**The action is in the HTTP method, NOT the URL.**

**6 Standard Methods:**

| Method | Purpose | Example | Idempotent? |
|--------|---------|---------|-------------|
| GET | Read/Retrieve | GET /users | Yes |
| POST | Create | POST /users | No |
| PUT | Update (full) | PUT /users/123 | Yes |
| PATCH | Update (partial) | PATCH /users/123 | No* |
| DELETE | Delete | DELETE /users/123 | Yes |
| OPTIONS | Get available methods | OPTIONS /users | Yes |

**Idempotent = Multiple identical requests have same effect as one request**

---

### CRUD Operations Mapped to HTTP Methods

**CRUD = Create, Read, Update, Delete**

| Operation | HTTP Method | Endpoint | Request Body | Response |
|-----------|-------------|----------|--------------|----------|
| Create | POST | /users | User data | 201 Created + new user |
| Read (all) | GET | /users | None | 200 OK + user list |
| Read (one) | GET | /users/123 | None | 200 OK + user |
| Update (full) | PUT | /users/123 | Full user data | 200 OK + updated user |
| Update (partial) | PATCH | /users/123 | Changed fields | 200 OK + updated user |
| Delete | DELETE | /users/123 | None | 204 No Content |

---

### Complete RESTful Endpoints Example

**Users Resource:**

```
GET    /users              List all users
GET    /users/123          Get user with ID 123
POST   /users              Create new user
PUT    /users/123          Update user 123 (full replacement)
PATCH  /users/123          Update user 123 (partial update)
DELETE /users/123          Delete user 123
```

**Posts Resource:**

```
GET    /posts              List all posts
GET    /posts/456          Get post with ID 456
POST   /posts              Create new post
PUT    /posts/456          Update post 456 (full)
PATCH  /posts/456          Update post 456 (partial)
DELETE /posts/456          Delete post 456
```

**Nested Resources (Relationships):**

```
GET    /users/123/posts         Get all posts by user 123
GET    /users/123/posts/456     Get post 456 by user 123
POST   /users/123/posts         Create post by user 123
DELETE /users/123/posts/456     Delete post 456 by user 123

GET    /posts/456/comments      Get comments on post 456
POST   /posts/456/comments      Create comment on post 456
DELETE /posts/456/comments/789  Delete comment 789 on post 456
```

---

### HTTP Status Codes (Standard Responses)

**Every response should have appropriate status code.**

**2xx - Success:**
```
200 OK              General success
201 Created         Resource created (POST)
204 No Content      Success but no body (DELETE)
```

**3xx - Redirection:**
```
301 Moved Permanently    Resource moved
304 Not Modified         Use cached version
```

**4xx - Client Errors:**
```
400 Bad Request          Invalid input
401 Unauthorized         Not authenticated
403 Forbidden            Not authorized
404 Not Found            Resource doesn't exist
409 Conflict             Resource already exists
422 Unprocessable Entity Validation failed
429 Too Many Requests    Rate limit exceeded
```

**5xx - Server Errors:**
```
500 Internal Server Error    Server crashed
502 Bad Gateway              Upstream server failed
503 Service Unavailable      Server overloaded
```

---

### Request/Response Format

**Standard: JSON**

**Request Example:**
```
POST /users
Content-Type: application/json

{
  "username": "alice",
  "email": "alice@example.com",
  "password": "SecurePass123!"
}
```

**Response Example:**
```
201 Created
Content-Type: application/json
Location: /users/123

{
  "id": 123,
  "username": "alice",
  "email": "alice@example.com",
  "created_at": "2024-12-23T10:30:00Z"
}
```

---

### Standard Response Structure

**Successful Response:**
```json
{
  "data": {
    "id": 123,
    "username": "alice",
    "email": "alice@example.com"
  }
}
```

**Error Response:**
```json
{
  "error": {
    "code": "VALIDATION_ERROR",
    "message": "Invalid email format",
    "details": [
      {
        "field": "email",
        "message": "Must be a valid email address"
      }
    ]
  }
}
```

**List Response (with pagination):**
```json
{
  "data": [
    { "id": 1, "username": "alice" },
    { "id": 2, "username": "bob" }
  ],
  "pagination": {
    "page": 1,
    "pageSize": 20,
    "total": 100,
    "totalPages": 5
  }
}
```

---

## RESTful Best Practices

### 1. Use Nouns, Not Verbs

**❌ Not RESTful:**
```
POST /createUser
GET /getUser/123
POST /deleteUser/123
GET /getAllUsers
```

**✅ RESTful:**
```
POST   /users       (create)
GET    /users/123   (read)
DELETE /users/123   (delete)
GET    /users       (read all)
```

**The HTTP method IS the verb.**

---

### 2. Use Plural Nouns for Collections

**❌ Inconsistent:**
```
/user/123
/users
```

**✅ Consistent:**
```
/users/123
/users
```

**Always plural, even for single resource.**

---

### 3. Use IDs in URL Path

**❌ Not RESTful:**
```
POST /users/delete
Body: { "id": 123 }

GET /users/get
Body: { "id": 123 }
```

**✅ RESTful:**
```
DELETE /users/123
(ID in URL path)

GET /users/123
(ID in URL path)
```

---

### 4. Use Query Parameters for Filtering

**Standard query parameters:**

```
Pagination:
GET /users?page=2&pageSize=20

Sorting:
GET /users?sortBy=created_at&sortOrder=desc

Filtering:
GET /users?role=admin&verified=true

Searching:
GET /users?search=alice

Combined:
GET /users?page=1&pageSize=20&role=admin&sortBy=username&sortOrder=asc
```

---

### 5. Nested Resources (2 Levels Max)

**✅ Good (1-2 levels):**
```
/users/123/posts
/posts/456/comments
```

**❌ Too deep (3+ levels):**
```
/users/123/posts/456/comments/789/likes
```

**Solution for deep nesting:**
```
Instead of:
/users/123/posts/456/comments/789/likes

Use:
/comments/789/likes
or
/likes?comment_id=789
```

---

### 6. Versioning

**Include API version in URL:**

```
/api/v1/users
/api/v1/posts

/api/v2/users  (breaking changes)
/api/v2/posts
```

**Why:** Allows old clients to keep using v1 while new clients use v2

---

### 7. Consistent Naming

**Use snake_case or camelCase (pick one):**

**snake_case:**
```json
{
  "user_id": 123,
  "created_at": "2024-12-23T10:30:00Z",
  "is_verified": true
}
```

**camelCase:**
```json
{
  "userId": 123,
  "createdAt": "2024-12-23T10:30:00Z",
  "isVerified": true
}
```

**Pick one, use everywhere.**

---

## Complete RESTful API Example

### Users API

```
Resource: Users

Endpoints:

1. List all users
   GET /api/v1/users?page=1&pageSize=20&role=admin
   Response: 200 OK + list of users

2. Get specific user
   GET /api/v1/users/123
   Response: 200 OK + user details
   Response: 404 Not Found (if doesn't exist)

3. Create user
   POST /api/v1/users
   Body: { "username": "alice", "email": "alice@example.com", "password": "..." }
   Response: 201 Created + new user
   Response: 409 Conflict (if already exists)

4. Update user (full)
   PUT /api/v1/users/123
   Body: { "username": "alice", "email": "alice@example.com", "role": "admin" }
   Response: 200 OK + updated user
   Response: 404 Not Found (if doesn't exist)

5. Update user (partial)
   PATCH /api/v1/users/123
   Body: { "email": "newemail@example.com" }
   Response: 200 OK + updated user

6. Delete user
   DELETE /api/v1/users/123
   Response: 204 No Content
   Response: 404 Not Found (if doesn't exist)

7. Get user's posts
   GET /api/v1/users/123/posts?page=1&pageSize=10
   Response: 200 OK + list of posts by user 123
```

---

### Posts API

```
Resource: Posts

Endpoints:

1. List all posts
   GET /api/v1/posts?page=1&pageSize=20&author_id=123
   Response: 200 OK + list of posts

2. Get specific post
   GET /api/v1/posts/456
   Response: 200 OK + post details

3. Create post
   POST /api/v1/posts
   Body: { "title": "My Post", "content": "...", "author_id": 123 }
   Response: 201 Created + new post

4. Update post
   PUT /api/v1/posts/456
   Body: { "title": "Updated Title", "content": "..." }
   Response: 200 OK + updated post

5. Delete post
   DELETE /api/v1/posts/456
   Response: 204 No Content

6. Get post comments
   GET /api/v1/posts/456/comments
   Response: 200 OK + list of comments

7. Create comment on post
   POST /api/v1/posts/456/comments
   Body: { "content": "Great post!", "author_id": 123 }
   Response: 201 Created + new comment
```

---

## RESTful vs Non-RESTful

### Non-RESTful (Bad Examples)

```
POST /createUser
POST /users/create
GET /getUser?id=123
POST /deleteUser
GET /user/123/delete
POST /updateUser
GET /users/all
POST /login/user
GET /getAllPostsByUser?userId=123
```

**Problems:**
- Verbs in URLs
- Inconsistent patterns
- Actions in paths
- Mixing singular/plural

---

### RESTful (Good Examples)

```
POST   /users              (create user)
GET    /users/123          (get user)
DELETE /users/123          (delete user)
PUT    /users/123          (update user)
GET    /users              (get all users)
POST   /auth/login         (login - special endpoint)
GET    /users/123/posts    (get posts by user)
```

**Benefits:**
- Clear patterns
- Predictable
- Standard HTTP methods
- Consistent structure

---

## Special Cases in REST

### 1. Authentication Endpoints

**Not strictly resources, but accepted:**

```
POST /auth/login
POST /auth/logout
POST /auth/register
POST /auth/refresh-token
POST /auth/forgot-password
POST /auth/reset-password
```

**Why exception:** Authentication is an action, not a resource

---

### 2. Search Endpoints

```
Option 1 (Query parameter):
GET /users?search=alice

Option 2 (Dedicated endpoint):
GET /search/users?q=alice

Option 3 (POST for complex search):
POST /users/search
Body: { "filters": {...}, "sort": {...} }
```

---

### 3. Bulk Operations

```
POST /users/bulk
Body: { "users": [...] }

DELETE /users/bulk
Body: { "ids": [123, 456, 789] }

PATCH /users/bulk
Body: { "updates": [...] }
```

---

### 4. Actions on Resources

**For actions that don't fit CRUD:**

```
POST /posts/456/publish
POST /posts/456/archive
POST /users/123/activate
POST /users/123/suspend
```

**Use POST + action verb (last resort)**

---

## Why REST Standard Matters

### 1. Predictability

**Developer sees:**
```
GET /products
```

**Developer knows:**
- This retrieves a list of products
- No body needed
- Will return 200 OK
- Response is array of products

**No documentation needed for basic understanding.**

---

### 2. Consistency

**All REST APIs work the same way:**
- GitHub API: GET /repos
- Twitter API: GET /tweets
- Your API: GET /users

**Same patterns everywhere.**

---

### 3. Tooling

**REST APIs work with:**
- Postman
- Swagger/OpenAPI
- HTTP clients
- Browsers (for GET requests)
- Curl
- Standard libraries

**Standard = better tooling support.**

---

### 4. Caching

**GET requests can be cached:**
- Browser caching
- CDN caching
- Proxy caching

**Standard HTTP caching works automatically.**

---

### 5. Scalability

**Stateless = easy to scale:**
- Add more servers
- Load balance requests
- No session synchronization needed (with tokens)

---

## Summary

### REST Core Principles

1. **Resource-based URLs** - Everything is a resource
2. **HTTP methods** - Use GET, POST, PUT, PATCH, DELETE correctly
3. **Status codes** - Use standard HTTP status codes
4. **Stateless** - Each request is independent
5. **JSON format** - Standard data format
6. **Predictable patterns** - Consistent URL structure

---

### RESTful URL Pattern

```
Collection:     /resources
Specific:       /resources/{id}
Nested:         /resources/{id}/nested
Filtering:      /resources?filter=value
Pagination:     /resources?page=1&pageSize=20
```

---

### HTTP Method Usage

```
GET    → Retrieve (read)
POST   → Create
PUT    → Update (full replacement)
PATCH  → Update (partial)
DELETE → Delete
```

---

### Standard Response Codes

```
200 OK
201 Created
204 No Content
400 Bad Request
401 Unauthorized
403 Forbidden
404 Not Found
500 Internal Server Error
```

---

**REST is a standard that makes APIs predictable, consistent, and easy to use. Following REST principles means your API works the way developers expect.**